# RSNA-2022-CSFD 解压 + 切 npy（Colab）

假设数据已放在 **`dataset/pretrain/RSNA-2022-CSFD/download/`**（例如 Kaggle 竞赛包或其中唯一顶层目录）。本笔记本会：

1. 创建与同目录平级的 **`unzip/`**、**`npy/`**
2. **解压与校验**：先运行 **「解压共用设置」**，再依次 **四个解压格**（同上四个 zip）；完成后可运行 **四个「文件数校验」格**，对照 zip 内文件条目数与 `unzip/` 下已落盘文件数是否一致。**Colab 本地盘很小**：`TMPDIR`/`TMP`/`TEMP` 指到 Drive 下 **`RSNA-2022-CSFD/.colab_tmp`**，**逐文件解压 + 进度条**。
3. 在 **`unzip/`** 有内容时从 `unzip/` 扫描体积；若 `unzip/` 为空而 **`download/`** 里已是解压好的目录，则回退为从 **`download/`** 扫描（无需再拷一遍）
4. **`pip install` 之后**先运行 **「了解目录结构」**；再运行 **「切 patch 共用」**，并**依次运行四个切 patch 格**（对应四个解压顶层目录）；所有 **`.npy` 直接写在 `npy/` 根目录**。

**切 npy 逻辑**与 `colab_abdomenct1k_preprocess.ipynb`、`preprocess/proc_spatial_sequence.py` 一致：

- 每个 3D 体积：**50** 个 npy（轴向 **z≥128**）或 **33** 个（**z<128**，`int(50/1.5)`）
- patch **128³**，`Resize` + `ScaleIntensityRangePercentiles(1, 99.9)`

**输入**：**DICOM** — 每个仅含 `.dcm`、且无子目录的叶子目录视为一个 3D 序列（与你当前解压结构一致：`train_part_*/…/StudyUID/*.dcm`、`test_images/…/StudyUID/*.dcm`）。**NIfTI** — 仅当存在体积 nii 时收集；**自动跳过路径中含 `segmentations/` 的 `.nii`**（竞赛为分割标注，非 CT，不做 patch）。可选 **`RSNA_INCLUDE_TEST_IMAGES=False`** 排除 `test_images`。跳过 **`__MACOSX`**。

**输出**：`.npy` 全部直接放在 **`npy/` 根目录**（不按 train_part 分子文件夹）；命名含 `RSNA-CSFD_StudyUID_…` 等，避免重名。

---

### 超大压缩包（例如约 189GB、仅一个 zip）与 Colab 本地盘（约一百～两百多 GB）

- **可以这样做的前提**：`*.zip` 放在 **Google Drive** 的 `download/` 里（路径形如 `/content/drive/MyDrive/...`），解压目标 **`unzip/` 也在同一块 Drive**。这样 **189GB 占用的是 Drive 配额**，**不是** Colab 根分区 `/` 的 200GB；逐文件解压只是把数据**流式写出到 Drive**，不会在本地再完整复制一份 189GB。
- **不行的情况**：若把 zip 放在 **`/content`**、或先 **`!cp` 到本地** 再解压，本地盘会被 **zip 体积 + 其它缓存** 撑满，与「只有 200 多 GB」矛盾。
- **仍可能 `Errno 28` 的原因**：挂载 Drive 的 FUSE **偶发**在本地做缓存；若多次失败，更稳妥的是 **在本机 / 大硬盘服务器上解压**，再同步到 Drive，或换 **Colab Pro / 更大临时盘** 的机型。
- **空间要算两处**：除 Colab 本地外，**解压后的未压缩总体积**常大于 zip，请确认 **Google Drive 剩余空间** 也足够（脚本会打印 zip 体积与磁盘用量）。

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os

BASE = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD"
DOWNLOAD = os.path.join(BASE, "download")
UNZIP_ROOT = os.path.join(BASE, "unzip")
NPY_ROOT = os.path.join(BASE, "npy")

for p in (BASE, DOWNLOAD, UNZIP_ROOT, NPY_ROOT):
    os.makedirs(p, exist_ok=True)

print("已创建/确认目录:")
print("  ", DOWNLOAD)
print("  ", UNZIP_ROOT)
print("  ", NPY_ROOT)

已创建/确认目录:
   /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/download
   /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip
   /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy


In [ ]:
# 解压共用设置（须先运行本格，再依次运行下面四个「解压 ①～④」格）
# TMP 指到 Drive；逐文件解压 + 进度；不做 testzip。
!pip install -q tqdm

import os
import shutil
import zipfile

DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/download"
UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip"
BASE = os.path.dirname(DOWNLOAD)
TMP_ON_DRIVE = os.path.join(BASE, ".colab_tmp")
os.makedirs(TMP_ON_DRIVE, exist_ok=True)
os.makedirs(UNZIP_ROOT, exist_ok=True)
for _k in ("TMPDIR", "TMP", "TEMP"):
    os.environ[_k] = TMP_ON_DRIVE

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None


def _print_disk():
    try:
        u = shutil.disk_usage("/")
        print(f"[分区 /] 可用 {u.free / 2**30:.2f} GB / 共 {u.total / 2**30:.2f} GB")
    except Exception:
        pass
    try:
        u2 = shutil.disk_usage(BASE)
        print(f"[Drive 数据目录] 可用 {u2.free / 2**30:.2f} GB（路径: {BASE}）")
    except Exception:
        pass
    print(f"[临时目录 TMPDIR] -> {TMP_ON_DRIVE}")


def _safe_extract_member(zf, member, dest_dir):
    dest_dir = os.path.realpath(dest_dir)
    out_path = os.path.realpath(os.path.join(dest_dir, member.filename))
    try:
        cp = os.path.commonpath([dest_dir, out_path])
    except ValueError:
        raise ValueError(f"非法路径: {member.filename}") from None
    if cp != dest_dir:
        raise ValueError(f"非法路径: {member.filename}")
    return zf.extract(member, dest_dir)


def extract_one_zip_progress(zpath, out_dir, label):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]
        total_sz = sum(m.file_size for m in members)
        n = len(members)
        print(f"    条目数: {n} 个文件，解压后约 {total_sz / 2**30:.2f} GiB（未压缩大小）")
        it = members
        if tqdm is not None:
            it = tqdm(members, desc=label[:48], unit="file", mininterval=1.0)
        done_b = 0
        for j, m in enumerate(it, start=1):
            _safe_extract_member(zf, m, out_dir)
            done_b += m.file_size
            if tqdm is None and (j == 1 or j % 500 == 0 or j == n):
                pct = 100.0 * done_b / total_sz if total_sz else 0
                print(f"    进度 {j}/{n} 文件 (~{pct:.1f}% 按未压缩字节)")
    return n


def _must_be_on_mounted_drive(path, label="zip"):
    rp = os.path.realpath(path)
    if not rp.startswith("/content/drive"):
        raise RuntimeError(
            f"{label} 必须位于已挂载的 Google Drive 下（路径应以 /content/drive 开头）。\n"
            f"当前: {rp}"
        )


def find_zip_in_download(download_root, basename):
    """在 download 根目录或子目录中按文件名查找 zip。"""
    bn = basename if basename.lower().endswith(".zip") else basename + ".zip"
    direct = os.path.join(download_root, bn)
    if os.path.isfile(direct):
        return direct
    for root, _, files in os.walk(download_root):
        if "__MACOSX" in root.replace("\\", "/"):
            continue
        if bn in files:
            return os.path.join(root, bn)
    raise FileNotFoundError(f"在 {download_root} 下未找到 {bn}")


def run_one_zip(zip_basename, title=None):
    """解压单个 zip 到 UNZIP_ROOT/<文件名去 .zip>/。"""
    bn = zip_basename if zip_basename.lower().endswith(".zip") else zip_basename + ".zip"
    zpath = find_zip_in_download(DOWNLOAD, bn)
    stem = os.path.splitext(os.path.basename(bn))[0]
    out_dir = os.path.join(UNZIP_ROOT, stem)
    rel = os.path.relpath(zpath, DOWNLOAD)
    t = title or bn
    print(f"\n=== {t} ===")
    print(f"    zip: {zpath}")
    print(f"    -> {out_dir}")
    _must_be_on_mounted_drive(zpath, "zip")
    _must_be_on_mounted_drive(out_dir, "解压目标目录")
    zbytes = os.path.getsize(zpath)
    print(f"    zip 文件大小（磁盘）: {zbytes / 2**30:.2f} GiB")
    extract_one_zip_progress(zpath, out_dir, rel)
    print("    OK")
    _print_disk()


def verify_extracted(zip_basename, label=""):
    """校验：zip 内「文件」条目数 vs Drive 上对应解压目录内递归文件数（排除 __MACOSX）。"""
    bn = zip_basename if zip_basename.lower().endswith(".zip") else zip_basename + ".zip"
    zpath = find_zip_in_download(DOWNLOAD, bn)
    stem = os.path.splitext(os.path.basename(bn))[0]
    out_dir = os.path.join(UNZIP_ROOT, stem)
    with zipfile.ZipFile(zpath, "r") as zf:
        n_zip = sum(1 for m in zf.infolist() if not m.is_dir())
    n_disk = 0
    if os.path.isdir(out_dir):
        for dp, _, fs in os.walk(out_dir):
            if "__MACOSX" in dp.replace("\\", "/"):
                continue
            n_disk += len(fs)
    ok = (n_zip == n_disk) and (n_zip > 0)
    tag = label or bn
    print(f"\n=== 文件数校验 {tag} ===")
    print(f"    zip 路径: {zpath}")
    print(f"    解压目录: {out_dir}")
    print(f"    zip 内文件条目数（非目录）: {n_zip}")
    print(f"    Drive 已解压文件数（递归）: {n_disk}")
    if ok:
        print("    结果: 通过（数量一致）")
    else:
        print("    结果: 未通过")
        if not os.path.isdir(out_dir):
            print("    说明: 解压目录不存在，可能尚未解压本包。")
        elif n_disk < n_zip:
            print("    说明: 磁盘文件数偏少，可能解压中断或未完成。")
        else:
            print("    说明: 磁盘文件数偏多，解压目录内可能有额外文件。")
    return ok


print("解压共用函数已就绪：run_one_zip('xxx.zip')；verify_extracted('xxx.zip')")
_print_disk()

### 解压 ①～④（分四次运行，减轻单次压力）

**先运行上一格「解压共用设置」**。再按需依次运行下面四格；某一格失败时可单独重跑该格（可先删掉 `unzip/` 下对应不完整子目录）。


In [ ]:
# 解压 ① rsna-2022-cervical-spine-fracture-detection.zip -> unzip/rsna-2022-cervical-spine-fracture-detection/
run_one_zip("rsna-2022-cervical-spine-fracture-detection.zip", title="① 主包")


In [ ]:
# 解压 ② train_part_1.zip -> unzip/train_part_1/
run_one_zip("train_part_1.zip", title="② train_part_1")


In [ ]:
# 解压 ③ train_part_2.zip -> unzip/train_part_2/
run_one_zip("train_part_2.zip", title="③ train_part_2")


In [ ]:
# 解压 ④ train_part_3.zip -> unzip/train_part_3/
run_one_zip("train_part_3.zip", title="④ train_part_3")


### 解压完整性校验（文件数量）

**须先运行「解压共用设置」格**（以载入 `verify_extracted`）。下面 **四个代码格** 分别对照：**zip 内非目录文件条目数** vs **`unzip/` 下对应子目录内递归文件数**（跳过 `__MACOSX`）。二者相等且大于 0 视为通过。


In [ ]:
# 校验 ① 主包：rsna-2022-cervical-spine-fracture-detection.zip
verify_extracted("rsna-2022-cervical-spine-fracture-detection.zip", label="① 主包")


In [ ]:
# 校验 ② train_part_1.zip
verify_extracted("train_part_1.zip", label="② train_part_1")


In [ ]:
# 校验 ③ train_part_2.zip
verify_extracted("train_part_2.zip", label="③ train_part_2")


In [ ]:
# 校验 ④ train_part_3.zip
verify_extracted("train_part_3.zip", label="④ train_part_3")


In [4]:
!pip install -q nibabel monai SimpleITK tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 14.5 MB/s eta 0:00:00


### 切 patch 前先跑：了解目录结构

下面一格会打印 **`DATA_ROOT`**（优先 `unzip/`，否则 `download/`）、**目录树（深度有限）**、**扩展名统计**、**NIfTI 样例路径**、**含 `.dcm` 的叶子目录样例及每层切片数**。把输出贴到对话里，便于判断是否与当前「递归 NIfTI + 叶子 DICOM 序列」逻辑一致。


In [ ]:
# 切 patch 之前：探查 DATA_ROOT 下的文件结构（输出给人 / 助手读即可）
import os
from collections import Counter

DRIVE_DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/download"
DRIVE_UNZIP = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip"


def get_volume_data_root():
    if os.path.isdir(DRIVE_UNZIP):
        for _, _, files in os.walk(DRIVE_UNZIP):
            if files:
                return DRIVE_UNZIP
    return DRIVE_DOWNLOAD


DATA_ROOT = get_volume_data_root()
SKIP = "__MACOSX"

print("=" * 72)
print("DATA_ROOT =", DATA_ROOT)
print("说明: 与后面切 patch 使用的根目录一致（unzip 非空则优先 unzip）。")
print("=" * 72)

if not os.path.isdir(DATA_ROOT):
    print("ERROR: 目录不存在，请先挂载 Drive 并确认路径。")
else:
    # --- 顶层一览 ---
    top = sorted(os.listdir(DATA_ROOT))
    print(f"\n[顶层] 共 {len(top)} 项（最多列 30 个）:")
    for name in top[:30]:
        p = os.path.join(DATA_ROOT, name)
        kind = "DIR " if os.path.isdir(p) else "FILE"
        print(f"  {kind}  {name}")
    if len(top) > 30:
        print(f"  ... 另有 {len(top) - 30} 项")

    # --- 扩展名统计（全树）---
    ext_cnt = Counter()
    n_files = 0
    n_dirs = 0
    max_depth = 0
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
        if SKIP in dirpath.replace("\\", "/"):
            continue
        n_dirs += len(dirnames)
        rel = os.path.relpath(dirpath, DATA_ROOT)
        depth = 0 if rel in (".", "") else rel.count(os.sep) + 1
        max_depth = max(max_depth, depth)
        for fn in filenames:
            n_files += 1
            lower = fn.lower()
            if "." in lower:
                ext_cnt[lower.rsplit(".", 1)[-1]] += 1
            else:
                ext_cnt["<no_ext>"] += 1

    print(f"\n[统计] 递归: 约 {n_dirs} 个子目录名出现、{n_files} 个文件；相对 DATA_ROOT 最大深度约 {max_depth}")
    print("[扩展名] 出现次数（降序，前 25）:")
    for ext, c in ext_cnt.most_common(25):
        print(f"  .{ext}: {c}")

    # --- 有限深度树（缩进 + 每级条数上限）---
    MAX_DEPTH = 4
    MAX_PER_LEVEL = 12

    def print_limited_tree(root, depth=0):
        indent = "  " * depth
        if depth > MAX_DEPTH:
            return
        try:
            names = [n for n in sorted(os.listdir(root)) if SKIP not in n and not n.startswith(".")]
        except OSError as e:
            print(indent + f"<无法列出: {e}>")
            return
        dirs = [n for n in names if os.path.isdir(os.path.join(root, n))]
        files = [n for n in names if n not in dirs]
        for name in dirs[:MAX_PER_LEVEL]:
            print(indent + "[DIR]  " + name + "/")
            print_limited_tree(os.path.join(root, name), depth + 1)
        if len(dirs) > MAX_PER_LEVEL:
            print(indent + f"... 还有 {len(dirs) - MAX_PER_LEVEL} 个子目录未显示")
        for name in files[:MAX_PER_LEVEL]:
            print(indent + "[FILE] " + name)
        if len(files) > MAX_PER_LEVEL:
            print(indent + f"... 还有 {len(files) - MAX_PER_LEVEL} 个文件未显示")

    print(f"\n[目录树] 深度<={MAX_DEPTH}，每级最多 {MAX_PER_LEVEL} 个目录 + {MAX_PER_LEVEL} 个文件（跳过隐藏与 __MACOSX）:")
    print_limited_tree(DATA_ROOT, 0)

    # --- NIfTI 路径样例 ---
    nii_samples = []
    for dirpath, _, files in os.walk(DATA_ROOT):
        if SKIP in dirpath.replace("\\", "/"):
            continue
        for f in files:
            if f.endswith(".nii.gz") or f.endswith(".nii"):
                nii_samples.append(os.path.relpath(os.path.join(dirpath, f), DATA_ROOT).replace("\\", "/"))
    nii_samples.sort()
    print(f"\n[NIfTI] 共 {len(nii_samples)} 个；前 15 条相对路径:")
    for p in nii_samples[:15]:
        print(" ", p)

    # --- DICOM：叶子目录（与预处理逻辑一致：有 .dcm 且无子目录）---
    leaf_dcm = []
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
        if SKIP in dirpath.replace("\\", "/"):
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if not dcms:
            continue
        if dirnames:
            continue
        leaf_dcm.append((dirpath, len(dcms)))

    leaf_dcm.sort(key=lambda x: x[0])
    print(f"\n[DICOM 叶子目录] 共 {len(leaf_dcm)} 个（无子目录、内含 .dcm）；与预处理将逐目录读成一个序列。")
    print("  前 12 条: 相对路径 | 该目录 .dcm 数")
    for dirpath, k in leaf_dcm[:12]:
        rel = os.path.relpath(dirpath, DATA_ROOT).replace("\\", "/")
        print(f"  {k:5d}  {rel}")

    # --- 含 dcm 但仍有子目录的目录（预处理不会当整序列）---
    mixed = []
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
        if SKIP in dirpath.replace("\\", "/"):
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if dcms and dirnames:
            mixed.append((dirpath, len(dcms), len(dirnames)))

    if mixed:
        print(f"\n[注意] {len(mixed)} 个目录「既有 .dcm 又有子目录」——当前脚本不会把它们当作单个 DICOM 序列；若你的数据在这里，需要改收集逻辑。")
        for dirpath, nd, ns in mixed[:8]:
            rel = os.path.relpath(dirpath, DATA_ROOT).replace("\\", "/")
            print(f"  {nd} dcm, {ns} subdirs  {rel}")

    # --- 可选：SimpleITK 试读前 2 个叶子序列的体素形状 ---
    try:
        import SimpleITK as sitk
    except ImportError:
        print("\n[试读] 未安装 SimpleITK，跳过体素形状探测（可先运行 pip 安装格）。")
    else:
        print("\n[试读 SimpleITK] 前 2 个叶子序列的 GetArrayFromImage().shape (z,y,x):")
        for dirpath, _ in leaf_dcm[:2]:
            try:
                r = sitk.ImageSeriesReader()
                names = r.GetGDCMSeriesFileNames(dirpath)
                r.SetFileNames(names)
                itk = r.Execute()
                arr = sitk.GetArrayFromImage(itk)
                rel = os.path.relpath(dirpath, DATA_ROOT).replace("\\", "/")
                print(f"  {rel}")
                print(f"    shape={tuple(arr.shape)}  slices={len(names)}")
            except Exception as e:
                rel = os.path.relpath(dirpath, DATA_ROOT).replace("\\", "/")
                print(f"  {rel}  -> 失败: {e}")

    print("\n" + "=" * 72)
    print("结构探查结束。确认无误后再运行「切 patch 共用」与四个切 patch 格。")
    print("=" * 72)


DATA_ROOT = /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip
说明: 与后面切 patch 使用的根目录一致（unzip 非空则优先 unzip）。

[顶层] 共 4 项（最多列 30 个）:
  DIR   rsna-2022-cervical-spine-fracture-detection
  DIR   train_part_1
  DIR   train_part_2
  DIR   train_part_3


KeyboardInterrupt: 

### 切 patch：共用 + 四分块

**先运行下一格「切 patch 共用」**（需已 `pip install nibabel monai SimpleITK tqdm`）。再**依次运行下面四格**，分别只处理 `unzip/` 下四个顶层目录之一。所有 `.npy` **直接保存在 `npy/` 根目录**（不按 part 建子目录）；DICOM 体积用 Study 目录名参与命名，避免冲突。


In [ ]:
# RSNA-2022-CSFD：切 patch 共用（须先运行本格，再依次运行下面四格）
# 所有 .npy 直接写在 npy/ 根目录，不建 train_part 等子文件夹。
import os
import random
import numpy as np
import nibabel as nib
import SimpleITK as sitk
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

DRIVE_UNZIP = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip"
DRIVE_SAVE_ROOT = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy"

PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0

RSNA_INCLUDE_TEST_IMAGES = True


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode="trilinear"),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume_zyx(image_zyx, patch_size_list, patch_num, save_root, start_index, tar_img_size, ds_name, vol_id):
    if len(image_zyx.shape) == 4:
        return []
    image = image_zyx
    n, patch_path_list = 0, []
    image_index = 0
    _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
    for i in range(_patch_num):
        n += 1
        patch_size = random.choice(patch_size_list)
        image_patch, _ = cut_patch(image, patch_size)
        image_patch = image_patch.transpose((2, 1, 0))
        image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
        save_name = os.path.join(save_root, "%s_%s_%s_%d.nii.gz" % (ds_name, vol_id, image_index, start_index + n))
        patch_path_list.append(save_name)
        np.save(save_name.replace(".nii.gz", ".npy"), image_patch)
    return patch_path_list


def cut_and_save_one_nii(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size, vol_id=None, ds_name=None):
    image, _affine = load_nii_data(image_file)
    path_parts = image_file.replace("\\", "/").split("/")
    if vol_id is None:
        ds_name = ds_name or (path_parts[-3] if len(path_parts) >= 3 else "RSNA-CSFD")
        nii_id = path_parts[-1].split(".nii.gz")[0].split(".nii")[0].split(".mha")[0]
    else:
        nii_id = vol_id
        ds_name = ds_name or "RSNA-CSFD"
    if len(image.shape) == 4:
        return []
    image = image.transpose((2, 1, 0))
    return cut_and_save_one_volume_zyx(image, patch_size_list, patch_num, save_root, start_index, tar_img_size, ds_name, nii_id)


def load_dicom_series_zyx(series_dir):
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(series_dir)
    if not names:
        return None
    reader.SetFileNames(names)
    itk = reader.Execute()
    arr = sitk.GetArrayFromImage(itk)
    return arr.astype(np.float32)


def collect_nii_files(root):
    out = []
    if not os.path.isdir(root):
        return out
    for dirpath, _, files in os.walk(root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm:
            continue
        if "segmentations" in norm.split("/"):
            continue
        for f in files:
            if f.endswith(".nii.gz") or f.endswith(".nii"):
                out.append(os.path.join(dirpath, f))
    return sorted(out)


def collect_dicom_series_dirs(root):
    series_dirs = []
    if not os.path.isdir(root):
        return series_dirs
    for dirpath, dirnames, filenames in os.walk(root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm:
            continue
        if not RSNA_INCLUDE_TEST_IMAGES and "test_images" in norm.split("/"):
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if not dcms:
            continue
        if dirnames:
            continue
        series_dirs.append(dirpath)
    return sorted(series_dirs)


def process_rsna_subfolder(subfolder_name, label=""):
    SCAN_ROOT = os.path.join(DRIVE_UNZIP, subfolder_name)
    if not os.path.isdir(SCAN_ROOT):
        print(f"[{label}] 跳过：目录不存在 -> {SCAN_ROOT}")
        return 0
    os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)
    nii_list = collect_nii_files(SCAN_ROOT)
    dicom_list = collect_dicom_series_dirs(SCAN_ROOT)
    print(f"\n========== [{label}] ==========")
    print(f"SCAN_ROOT = {SCAN_ROOT}")
    print(f"NIfTI: {len(nii_list)} | DICOM 序列: {len(dicom_list)}")
    if len(nii_list) == 0 and len(dicom_list) == 0:
        print("未找到 .nii 或 DICOM 叶子序列。")
        return 0
    patch_list_all = []
    for i, path in enumerate(tqdm(nii_list, desc=f"{label} NIfTI")):
        rel = os.path.relpath(path, SCAN_ROOT).replace("\\", "/")
        vol_id = rel.replace("/", "_").replace(".nii.gz", "").replace(".nii", "")
        pl = cut_and_save_one_nii(
            path, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE,
            vol_id=vol_id, ds_name="RSNA-CSFD",
        )
        patch_list_all += pl
    for i, series_dir in enumerate(tqdm(dicom_list, desc=f"{label} DICOM")):
        try:
            vol = load_dicom_series_zyx(series_dir)
        except Exception as e:
            print(f"跳过: {series_dir}\n  {e}")
            continue
        if vol is None:
            continue
        vol_id = os.path.basename(series_dir.rstrip(os.sep))
        pl = cut_and_save_one_volume_zyx(
            vol, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE, "RSNA-CSFD", vol_id,
        )
        patch_list_all += pl
    print(f"[{label}] 本块写入 npy 数: {len(patch_list_all)}（目录: {DRIVE_SAVE_ROOT} ，无子文件夹）")
    return len(patch_list_all)


print("已就绪：process_rsna_subfolder('顶层文件夹名', '显示标签')")


已就绪：process_rsna_subfolder('顶层文件夹名', '显示标签')


In [ ]:
# 切 patch ① 主包（unzip/rsna-2022-cervical-spine-fracture-detection/）
process_rsna_subfolder("rsna-2022-cervical-spine-fracture-detection", "① 主包")



========== [① 主包] ==========
SCAN_ROOT = /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/rsna-2022-cervical-spine-fracture-detection
NIfTI: 0 | DICOM 序列: 3


① 主包 NIfTI: 0it [00:00, ?it/s]
① 主包 DICOM: 100%|██████████| 3/3 [02:49<00:00, 56.49s/it]

[① 主包] 本块写入 npy 数: 150（目录: /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy ，无子文件夹）


150

In [ ]:
# 切 patch ② train_part_1
process_rsna_subfolder("train_part_1", "② train_part_1")



========== [② train_part_1] ==========
SCAN_ROOT = /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/train_part_1
NIfTI: 0 | DICOM 序列: 690


② train_part_1 NIfTI: 0it [00:00, ?it/s]
② train_part_1 DICOM: 100%|██████████| 690/690 [7:49:30<00:00, 40.83s/it]

[② train_part_1] 本块写入 npy 数: 34466（目录: /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy ，无子文件夹）


34466

In [ ]:
# 切 patch ③ train_part_2
process_rsna_subfolder("train_part_2", "③ train_part_2")



========== [③ train_part_2] ==========
SCAN_ROOT = /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/train_part_2
NIfTI: 0 | DICOM 序列: 690


③ train_part_2 NIfTI: 0it [00:00, ?it/s]
③ train_part_2 DICOM:  18%|█▊        | 123/690 [3:46:28<14:19:12, 90.92s/it] /usr/local/lib/python3.12/dist-packages/monai/transforms/intensity/array.py:1001: Warning: Divide by zero (a_min == a_max)
  warn("Divide by zero (a_min == a_max)", Warning)
③ train_part_2 DICOM: 100%|██████████| 690/690 [22:27:39<00:00, 117.19s/it]

[③ train_part_2] 本块写入 npy 数: 34483（目录: /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy ，无子文件夹）


34483

In [ ]:
# 切 patch ④ train_part_3
process_rsna_subfolder("train_part_3", "④ train_part_3")



========== [④ train_part_3] ==========
SCAN_ROOT = /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/train_part_3
NIfTI: 0 | DICOM 序列: 639


④ train_part_3 NIfTI: 0it [00:00, ?it/s]
④ train_part_3 DICOM:   5%|▍         | 31/639 [18:36<6:21:47, 37.68s/it]/usr/local/lib/python3.12/dist-packages/monai/transforms/intensity/array.py:1001: Warning: Divide by zero (a_min == a_max)
  warn("Divide by zero (a_min == a_max)", Warning)
④ train_part_3 DICOM:  77%|███████▋  | 489/639 [5:05:30<2:40:19, 64.13s/it]

In [ ]:
# （可选）生成 train_patch_spatial.txt 供 pretrain 使用
import os

DRIVE_SAVE_ROOT = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy"
list_save_dir = os.path.join(os.path.dirname(DRIVE_SAVE_ROOT), "pretrain_lists")
os.makedirs(list_save_dir, exist_ok=True)
npy_files = sorted([os.path.join(DRIVE_SAVE_ROOT, f) for f in os.listdir(DRIVE_SAVE_ROOT) if f.endswith(".npy")])
with open(os.path.join(list_save_dir, "train_patch_spatial.txt"), "w") as f:
    f.write("\n".join(npy_files))
print(f"train_patch_spatial.txt 已保存: {list_save_dir}（共 {len(npy_files)} 行）")

In [7]:
# train_part_3：完整 / 部分 / 未切 —— 全部列出（对比 npy/）
!pip install -q nibabel

import os
from collections import defaultdict

import nibabel as nib

DRIVE_UNZIP = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip"
DRIVE_SAVE_ROOT = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/npy"
SUB = "train_part_3"
PATCH_NUM = 50
RSNA_INCLUDE_TEST_IMAGES = True

SCAN_ROOT = os.path.join(DRIVE_UNZIP, SUB)


def expected_patches_from_z(z):
    return int(PATCH_NUM / 1.5) if z < 128 else PATCH_NUM


def collect_nii_files(root):
    out = []
    if not os.path.isdir(root):
        return out
    for dirpath, _, files in os.walk(root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm or "segmentations" in norm.split("/"):
            continue
        for f in files:
            if f.endswith(".nii.gz") or f.endswith(".nii"):
                out.append(os.path.join(dirpath, f))
    return sorted(out)


def collect_dicom_series_dirs(root):
    out = []
    if not os.path.isdir(root):
        return out
    for dirpath, dirnames, filenames in os.walk(root):
        norm = dirpath.replace("\\", "/")
        if "__MACOSX" in norm:
            continue
        if not RSNA_INCLUDE_TEST_IMAGES and "test_images" in norm.split("/"):
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if not dcms or dirnames:
            continue
        out.append(dirpath)
    return sorted(out)


def nii_z_dim(path):
    data = nib.load(path).get_fdata()
    if data.ndim != 3:
        return None
    return int(data.shape[2])


def parse_npy(fn):
    if not (fn.endswith(".npy") and fn.startswith("RSNA-CSFD_")):
        return None
    stem = fn[:-4]
    if "_0_" not in stem:
        return None
    left, num_s = stem.rsplit("_0_", 1)
    if not left.startswith("RSNA-CSFD_"):
        return None
    try:
        k = int(num_s)
    except ValueError:
        return None
    vol_id = left[len("RSNA-CSFD_") :]
    return vol_id, k


by_vol = defaultdict(set)
for fn in os.listdir(DRIVE_SAVE_ROOT):
    p = parse_npy(fn)
    if p:
        by_vol[p[0]].add(p[1])

if not os.path.isdir(SCAN_ROOT):
    raise FileNotFoundError(SCAN_ROOT)

volumes = []
for path in collect_nii_files(SCAN_ROOT):
    rel = os.path.relpath(path, SCAN_ROOT).replace("\\", "/")
    vid = rel.replace("/", "_").replace(".nii.gz", "").replace(".nii", "")
    z = nii_z_dim(path)
    if z is not None:
        volumes.append((vid, expected_patches_from_z(z)))

for series_dir in collect_dicom_series_dirs(SCAN_ROOT):
    vid = os.path.basename(series_dir.rstrip(os.sep))
    z = len([f for f in os.listdir(series_dir) if f.lower().endswith(".dcm")])
    volumes.append((vid, expected_patches_from_z(z)))

done, partial, none = [], [], []

for vid, exp in volumes:
    need = set(range(1, exp + 1))
    got = by_vol.get(vid, set())
    if got == need:
        done.append((vid, exp))
    elif len(got) == 0:
        none.append((vid, exp))
    else:
        partial.append((vid, len(got), exp, sorted(got)))

print("=" * 72)
print(f"train_part_3 | SCAN_ROOT={SCAN_ROOT}")
print(f"应切分体积总数: {len(volumes)}")
print(f"已全部切完: {len(done)}")
print(f"只切了一部分: {len(partial)}")
print(f"完全未切: {len(none)}")
print("=" * 72)

print("\n【已全部切完 — 全部列出】vol_id, 应有patch数")
for vid, exp in done:
    print(vid, exp)

print("\n【只切了一部分 — 全部列出】vol_id, 已有数, 应有数, 已有编号排序列表")
for vid, ng, exp, ks in partial:
    print(vid, ng, exp, ks)

print("\n【完全未切 — 全部列出】vol_id, 应有patch数")
for vid, exp in none:
    print(vid, exp)

train_part_3 | SCAN_ROOT=/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/train_part_3
应切分体积总数: 639
已全部切完: 489
只切了一部分: 0
完全未切: 150

【已全部切完 — 全部列出】vol_id, 应有patch数
1.2.826.0.1.3680043.22310 50
1.2.826.0.1.3680043.22358 50
1.2.826.0.1.3680043.22361 50
1.2.826.0.1.3680043.22392 50
1.2.826.0.1.3680043.22430 50
1.2.826.0.1.3680043.22436 50
1.2.826.0.1.3680043.22438 50
1.2.826.0.1.3680043.22444 50
1.2.826.0.1.3680043.22447 50
1.2.826.0.1.3680043.22477 50
1.2.826.0.1.3680043.22478 50
1.2.826.0.1.3680043.22510 50
1.2.826.0.1.3680043.22517 50
1.2.826.0.1.3680043.22538 50
1.2.826.0.1.3680043.22543 50
1.2.826.0.1.3680043.22548 50
1.2.826.0.1.3680043.22560 50
1.2.826.0.1.3680043.22575 50
1.2.826.0.1.3680043.22596 50
1.2.826.0.1.3680043.22643 50
1.2.826.0.1.3680043.22669 50
1.2.826.0.1.3680043.22672 50
1.2.826.0.1.3680043.22675 50
1.2.826.0.1.3680043.22678 50
1.2.826.0.1.3680043.22691 50
1.2.826.0.1.3680043.22701 50
1.2.826.0.1.3680043.22711 50
1.2.826.0.1.3680043.22717 50
1.2.826.0.1.3

In [9]:
# 根据「train_part_3 校验」粘贴文本，把缺少的体积复制到 unzip/missing/（复制，不移动）
import os
import re
import shutil

# ========= 把你的整段输出粘贴在下面三引号之间 =========
PASTE = r"""
========================================================================
train_part_3 | SCAN_ROOT=/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/train_part_3
应切分体积总数: 639
已全部切完: 489
只切了一部分: 0
完全未切: 150
========================================================================

【已全部切完 — 全部列出】vol_id, 应有patch数
1.2.826.0.1.3680043.22310 50
1.2.826.0.1.3680043.22358 50
1.2.826.0.1.3680043.22361 50
1.2.826.0.1.3680043.22392 50
1.2.826.0.1.3680043.22430 50
1.2.826.0.1.3680043.22436 50
1.2.826.0.1.3680043.22438 50
1.2.826.0.1.3680043.22444 50
1.2.826.0.1.3680043.22447 50
1.2.826.0.1.3680043.22477 50
1.2.826.0.1.3680043.22478 50
1.2.826.0.1.3680043.22510 50
1.2.826.0.1.3680043.22517 50
1.2.826.0.1.3680043.22538 50
1.2.826.0.1.3680043.22543 50
1.2.826.0.1.3680043.22548 50
1.2.826.0.1.3680043.22560 50
1.2.826.0.1.3680043.22575 50
1.2.826.0.1.3680043.22596 50
1.2.826.0.1.3680043.22643 50
1.2.826.0.1.3680043.22669 50
1.2.826.0.1.3680043.22672 50
1.2.826.0.1.3680043.22675 50
1.2.826.0.1.3680043.22678 50
1.2.826.0.1.3680043.22691 50
1.2.826.0.1.3680043.22701 50
1.2.826.0.1.3680043.22711 50
1.2.826.0.1.3680043.22717 50
1.2.826.0.1.3680043.22755 50
1.2.826.0.1.3680043.22757 50
1.2.826.0.1.3680043.22759 50
1.2.826.0.1.3680043.22780 50
1.2.826.0.1.3680043.22793 50
1.2.826.0.1.3680043.22804 50
1.2.826.0.1.3680043.22806 50
1.2.826.0.1.3680043.22808 50
1.2.826.0.1.3680043.22829 50
1.2.826.0.1.3680043.22841 50
1.2.826.0.1.3680043.22885 50
1.2.826.0.1.3680043.22889 50
1.2.826.0.1.3680043.22926 50
1.2.826.0.1.3680043.22930 50
1.2.826.0.1.3680043.22944 50
1.2.826.0.1.3680043.22950 50
1.2.826.0.1.3680043.22968 50
1.2.826.0.1.3680043.22980 50
1.2.826.0.1.3680043.22993 50
1.2.826.0.1.3680043.23052 50
1.2.826.0.1.3680043.23099 50
1.2.826.0.1.3680043.23104 50
1.2.826.0.1.3680043.23126 50
1.2.826.0.1.3680043.23168 50
1.2.826.0.1.3680043.23212 50
1.2.826.0.1.3680043.23216 50
1.2.826.0.1.3680043.23251 50
1.2.826.0.1.3680043.23252 50
1.2.826.0.1.3680043.23257 50
1.2.826.0.1.3680043.23260 50
1.2.826.0.1.3680043.23261 50
1.2.826.0.1.3680043.23325 50
1.2.826.0.1.3680043.23353 50
1.2.826.0.1.3680043.23355 50
1.2.826.0.1.3680043.23356 50
1.2.826.0.1.3680043.23361 50
1.2.826.0.1.3680043.23400 50
1.2.826.0.1.3680043.23410 50
1.2.826.0.1.3680043.23422 50
1.2.826.0.1.3680043.23431 50
1.2.826.0.1.3680043.23432 50
1.2.826.0.1.3680043.23449 50
1.2.826.0.1.3680043.23474 50
1.2.826.0.1.3680043.23475 50
1.2.826.0.1.3680043.23509 50
1.2.826.0.1.3680043.23527 50
1.2.826.0.1.3680043.23528 50
1.2.826.0.1.3680043.23536 50
1.2.826.0.1.3680043.23543 50
1.2.826.0.1.3680043.23597 50
1.2.826.0.1.3680043.23611 50
1.2.826.0.1.3680043.23633 50
1.2.826.0.1.3680043.23648 50
1.2.826.0.1.3680043.23658 50
1.2.826.0.1.3680043.23660 50
1.2.826.0.1.3680043.23672 50
1.2.826.0.1.3680043.23697 50
1.2.826.0.1.3680043.23715 50
1.2.826.0.1.3680043.23758 50
1.2.826.0.1.3680043.23765 50
1.2.826.0.1.3680043.23794 50
1.2.826.0.1.3680043.23809 50
1.2.826.0.1.3680043.23817 50
1.2.826.0.1.3680043.23840 50
1.2.826.0.1.3680043.23846 50
1.2.826.0.1.3680043.23847 50
1.2.826.0.1.3680043.23848 50
1.2.826.0.1.3680043.23858 50
1.2.826.0.1.3680043.23860 50
1.2.826.0.1.3680043.23888 50
1.2.826.0.1.3680043.23904 50
1.2.826.0.1.3680043.23916 50
1.2.826.0.1.3680043.23923 50
1.2.826.0.1.3680043.23944 50
1.2.826.0.1.3680043.23957 50
1.2.826.0.1.3680043.23968 50
1.2.826.0.1.3680043.23978 50
1.2.826.0.1.3680043.23986 50
1.2.826.0.1.3680043.24007 50
1.2.826.0.1.3680043.24010 50
1.2.826.0.1.3680043.24013 50
1.2.826.0.1.3680043.24018 50
1.2.826.0.1.3680043.24045 50
1.2.826.0.1.3680043.24049 50
1.2.826.0.1.3680043.24107 50
1.2.826.0.1.3680043.24115 50
1.2.826.0.1.3680043.24131 50
1.2.826.0.1.3680043.24140 50
1.2.826.0.1.3680043.24149 50
1.2.826.0.1.3680043.24170 50
1.2.826.0.1.3680043.24178 50
1.2.826.0.1.3680043.24206 50
1.2.826.0.1.3680043.24214 50
1.2.826.0.1.3680043.24264 50
1.2.826.0.1.3680043.24281 50
1.2.826.0.1.3680043.24284 50
1.2.826.0.1.3680043.24289 50
1.2.826.0.1.3680043.24307 50
1.2.826.0.1.3680043.24316 50
1.2.826.0.1.3680043.24317 50
1.2.826.0.1.3680043.24325 50
1.2.826.0.1.3680043.24326 50
1.2.826.0.1.3680043.24327 50
1.2.826.0.1.3680043.24343 50
1.2.826.0.1.3680043.24385 50
1.2.826.0.1.3680043.24389 50
1.2.826.0.1.3680043.24415 50
1.2.826.0.1.3680043.24418 50
1.2.826.0.1.3680043.24445 50
1.2.826.0.1.3680043.24477 50
1.2.826.0.1.3680043.24508 50
1.2.826.0.1.3680043.24562 50
1.2.826.0.1.3680043.24570 50
1.2.826.0.1.3680043.24593 50
1.2.826.0.1.3680043.24595 50
1.2.826.0.1.3680043.24606 50
1.2.826.0.1.3680043.24615 50
1.2.826.0.1.3680043.24617 50
1.2.826.0.1.3680043.24639 50
1.2.826.0.1.3680043.24643 50
1.2.826.0.1.3680043.24673 50
1.2.826.0.1.3680043.24691 50
1.2.826.0.1.3680043.24704 50
1.2.826.0.1.3680043.24761 50
1.2.826.0.1.3680043.24772 50
1.2.826.0.1.3680043.24857 50
1.2.826.0.1.3680043.24878 50
1.2.826.0.1.3680043.24891 50
1.2.826.0.1.3680043.24918 50
1.2.826.0.1.3680043.24930 50
1.2.826.0.1.3680043.24944 50
1.2.826.0.1.3680043.24946 50
1.2.826.0.1.3680043.24953 50
1.2.826.0.1.3680043.24955 50
1.2.826.0.1.3680043.24962 50
1.2.826.0.1.3680043.24974 50
1.2.826.0.1.3680043.24988 50
1.2.826.0.1.3680043.24989 50
1.2.826.0.1.3680043.25035 50
1.2.826.0.1.3680043.25071 50
1.2.826.0.1.3680043.25080 50
1.2.826.0.1.3680043.25093 50
1.2.826.0.1.3680043.25103 50
1.2.826.0.1.3680043.25130 50
1.2.826.0.1.3680043.25144 50
1.2.826.0.1.3680043.25154 50
1.2.826.0.1.3680043.25161 50
1.2.826.0.1.3680043.25172 50
1.2.826.0.1.3680043.25174 50
1.2.826.0.1.3680043.25191 50
1.2.826.0.1.3680043.25198 50
1.2.826.0.1.3680043.25220 50
1.2.826.0.1.3680043.25238 50
1.2.826.0.1.3680043.25245 50
1.2.826.0.1.3680043.25278 50
1.2.826.0.1.3680043.25289 50
1.2.826.0.1.3680043.25301 50
1.2.826.0.1.3680043.25319 50
1.2.826.0.1.3680043.25339 50
1.2.826.0.1.3680043.25352 50
1.2.826.0.1.3680043.25356 50
1.2.826.0.1.3680043.25367 50
1.2.826.0.1.3680043.25377 50
1.2.826.0.1.3680043.25434 50
1.2.826.0.1.3680043.25435 50
1.2.826.0.1.3680043.25445 50
1.2.826.0.1.3680043.25493 50
1.2.826.0.1.3680043.25509 50
1.2.826.0.1.3680043.25515 50
1.2.826.0.1.3680043.25548 50
1.2.826.0.1.3680043.25572 50
1.2.826.0.1.3680043.25600 50
1.2.826.0.1.3680043.25616 50
1.2.826.0.1.3680043.25638 50
1.2.826.0.1.3680043.25651 50
1.2.826.0.1.3680043.25669 50
1.2.826.0.1.3680043.25704 50
1.2.826.0.1.3680043.25722 50
1.2.826.0.1.3680043.25770 50
1.2.826.0.1.3680043.25772 50
1.2.826.0.1.3680043.25812 50
1.2.826.0.1.3680043.25825 50
1.2.826.0.1.3680043.25833 50
1.2.826.0.1.3680043.25834 50
1.2.826.0.1.3680043.25838 50
1.2.826.0.1.3680043.25867 50
1.2.826.0.1.3680043.25883 50
1.2.826.0.1.3680043.25891 50
1.2.826.0.1.3680043.25917 50
1.2.826.0.1.3680043.25919 50
1.2.826.0.1.3680043.25956 50
1.2.826.0.1.3680043.25963 50
1.2.826.0.1.3680043.25970 50
1.2.826.0.1.3680043.25987 50
1.2.826.0.1.3680043.26024 50
1.2.826.0.1.3680043.26031 50
1.2.826.0.1.3680043.26034 50
1.2.826.0.1.3680043.26035 50
1.2.826.0.1.3680043.26040 50
1.2.826.0.1.3680043.26052 50
1.2.826.0.1.3680043.26068 50
1.2.826.0.1.3680043.26077 50
1.2.826.0.1.3680043.26097 50
1.2.826.0.1.3680043.26110 50
1.2.826.0.1.3680043.26121 50
1.2.826.0.1.3680043.26170 50
1.2.826.0.1.3680043.26177 50
1.2.826.0.1.3680043.26179 50
1.2.826.0.1.3680043.26187 50
1.2.826.0.1.3680043.26207 50
1.2.826.0.1.3680043.26217 50
1.2.826.0.1.3680043.26219 50
1.2.826.0.1.3680043.26231 50
1.2.826.0.1.3680043.26244 50
1.2.826.0.1.3680043.26293 50
1.2.826.0.1.3680043.26303 50
1.2.826.0.1.3680043.26306 50
1.2.826.0.1.3680043.26315 50
1.2.826.0.1.3680043.26350 50
1.2.826.0.1.3680043.26389 50
1.2.826.0.1.3680043.26401 50
1.2.826.0.1.3680043.26440 50
1.2.826.0.1.3680043.26442 50
1.2.826.0.1.3680043.26469 50
1.2.826.0.1.3680043.26492 50
1.2.826.0.1.3680043.26498 50
1.2.826.0.1.3680043.26500 50
1.2.826.0.1.3680043.26514 50
1.2.826.0.1.3680043.26534 50
1.2.826.0.1.3680043.26536 50
1.2.826.0.1.3680043.26574 50
1.2.826.0.1.3680043.26618 50
1.2.826.0.1.3680043.26649 50
1.2.826.0.1.3680043.26665 50
1.2.826.0.1.3680043.26676 50
1.2.826.0.1.3680043.26740 50
1.2.826.0.1.3680043.26769 50
1.2.826.0.1.3680043.26781 50
1.2.826.0.1.3680043.26791 50
1.2.826.0.1.3680043.26801 50
1.2.826.0.1.3680043.26802 50
1.2.826.0.1.3680043.26820 50
1.2.826.0.1.3680043.26830 50
1.2.826.0.1.3680043.26851 50
1.2.826.0.1.3680043.26869 50
1.2.826.0.1.3680043.26898 50
1.2.826.0.1.3680043.26917 50
1.2.826.0.1.3680043.26933 50
1.2.826.0.1.3680043.26950 50
1.2.826.0.1.3680043.26958 50
1.2.826.0.1.3680043.26959 50
1.2.826.0.1.3680043.26969 50
1.2.826.0.1.3680043.26979 50
1.2.826.0.1.3680043.26990 50
1.2.826.0.1.3680043.26992 50
1.2.826.0.1.3680043.27016 50
1.2.826.0.1.3680043.27027 50
1.2.826.0.1.3680043.27028 50
1.2.826.0.1.3680043.27030 50
1.2.826.0.1.3680043.27043 50
1.2.826.0.1.3680043.27075 50
1.2.826.0.1.3680043.27079 50
1.2.826.0.1.3680043.27100 50
1.2.826.0.1.3680043.27121 50
1.2.826.0.1.3680043.27180 50
1.2.826.0.1.3680043.27218 50
1.2.826.0.1.3680043.27225 50
1.2.826.0.1.3680043.27228 50
1.2.826.0.1.3680043.27262 50
1.2.826.0.1.3680043.27275 50
1.2.826.0.1.3680043.27282 50
1.2.826.0.1.3680043.27284 50
1.2.826.0.1.3680043.27292 50
1.2.826.0.1.3680043.27299 50
1.2.826.0.1.3680043.27316 50
1.2.826.0.1.3680043.27325 50
1.2.826.0.1.3680043.27329 50
1.2.826.0.1.3680043.27360 50
1.2.826.0.1.3680043.27361 50
1.2.826.0.1.3680043.27372 50
1.2.826.0.1.3680043.27390 50
1.2.826.0.1.3680043.27395 50
1.2.826.0.1.3680043.27398 50
1.2.826.0.1.3680043.27411 50
1.2.826.0.1.3680043.27426 50
1.2.826.0.1.3680043.27436 50
1.2.826.0.1.3680043.27440 50
1.2.826.0.1.3680043.27523 50
1.2.826.0.1.3680043.27562 50
1.2.826.0.1.3680043.27565 50
1.2.826.0.1.3680043.27569 50
1.2.826.0.1.3680043.27578 50
1.2.826.0.1.3680043.27583 50
1.2.826.0.1.3680043.27607 50
1.2.826.0.1.3680043.27622 50
1.2.826.0.1.3680043.27670 50
1.2.826.0.1.3680043.27683 50
1.2.826.0.1.3680043.27714 50
1.2.826.0.1.3680043.27733 50
1.2.826.0.1.3680043.27745 50
1.2.826.0.1.3680043.27746 50
1.2.826.0.1.3680043.27748 50
1.2.826.0.1.3680043.27752 50
1.2.826.0.1.3680043.27770 50
1.2.826.0.1.3680043.27777 50
1.2.826.0.1.3680043.27793 50
1.2.826.0.1.3680043.27828 50
1.2.826.0.1.3680043.27830 50
1.2.826.0.1.3680043.27837 50
1.2.826.0.1.3680043.27850 50
1.2.826.0.1.3680043.27863 50
1.2.826.0.1.3680043.27962 50
1.2.826.0.1.3680043.27975 50
1.2.826.0.1.3680043.27996 50
1.2.826.0.1.3680043.28019 50
1.2.826.0.1.3680043.28025 50
1.2.826.0.1.3680043.28055 50
1.2.826.0.1.3680043.28122 50
1.2.826.0.1.3680043.28153 50
1.2.826.0.1.3680043.28163 50
1.2.826.0.1.3680043.28207 50
1.2.826.0.1.3680043.28215 50
1.2.826.0.1.3680043.28233 50
1.2.826.0.1.3680043.28252 50
1.2.826.0.1.3680043.28254 50
1.2.826.0.1.3680043.28273 50
1.2.826.0.1.3680043.28282 50
1.2.826.0.1.3680043.28296 50
1.2.826.0.1.3680043.28297 50
1.2.826.0.1.3680043.28304 50
1.2.826.0.1.3680043.28319 50
1.2.826.0.1.3680043.28321 50
1.2.826.0.1.3680043.28323 50
1.2.826.0.1.3680043.28327 50
1.2.826.0.1.3680043.28329 50
1.2.826.0.1.3680043.28344 50
1.2.826.0.1.3680043.28346 50
1.2.826.0.1.3680043.28355 50
1.2.826.0.1.3680043.28363 50
1.2.826.0.1.3680043.28411 50
1.2.826.0.1.3680043.28414 50
1.2.826.0.1.3680043.28480 50
1.2.826.0.1.3680043.28485 50
1.2.826.0.1.3680043.28502 50
1.2.826.0.1.3680043.28521 50
1.2.826.0.1.3680043.28523 50
1.2.826.0.1.3680043.28527 50
1.2.826.0.1.3680043.28547 50
1.2.826.0.1.3680043.28580 50
1.2.826.0.1.3680043.28606 50
1.2.826.0.1.3680043.28610 50
1.2.826.0.1.3680043.28618 50
1.2.826.0.1.3680043.28620 50
1.2.826.0.1.3680043.28637 50
1.2.826.0.1.3680043.28657 50
1.2.826.0.1.3680043.28665 50
1.2.826.0.1.3680043.28705 50
1.2.826.0.1.3680043.28710 50
1.2.826.0.1.3680043.28711 50
1.2.826.0.1.3680043.28731 50
1.2.826.0.1.3680043.28752 50
1.2.826.0.1.3680043.28753 50
1.2.826.0.1.3680043.28780 50
1.2.826.0.1.3680043.28794 50
1.2.826.0.1.3680043.28805 50
1.2.826.0.1.3680043.28824 50
1.2.826.0.1.3680043.28825 50
1.2.826.0.1.3680043.28832 50
1.2.826.0.1.3680043.28857 50
1.2.826.0.1.3680043.28865 50
1.2.826.0.1.3680043.28873 50
1.2.826.0.1.3680043.28890 50
1.2.826.0.1.3680043.28892 50
1.2.826.0.1.3680043.28904 50
1.2.826.0.1.3680043.28913 50
1.2.826.0.1.3680043.28934 50
1.2.826.0.1.3680043.28968 50
1.2.826.0.1.3680043.28978 50
1.2.826.0.1.3680043.28990 50
1.2.826.0.1.3680043.29015 50
1.2.826.0.1.3680043.29035 50
1.2.826.0.1.3680043.29045 50
1.2.826.0.1.3680043.29047 50
1.2.826.0.1.3680043.29093 50
1.2.826.0.1.3680043.29100 50
1.2.826.0.1.3680043.29132 50
1.2.826.0.1.3680043.29140 50
1.2.826.0.1.3680043.29142 50
1.2.826.0.1.3680043.29183 50
1.2.826.0.1.3680043.29194 50
1.2.826.0.1.3680043.29213 50
1.2.826.0.1.3680043.29218 50
1.2.826.0.1.3680043.29243 50
1.2.826.0.1.3680043.29258 50
1.2.826.0.1.3680043.29261 50
1.2.826.0.1.3680043.29287 50
1.2.826.0.1.3680043.29329 50
1.2.826.0.1.3680043.29330 50
1.2.826.0.1.3680043.29351 50
1.2.826.0.1.3680043.29378 50
1.2.826.0.1.3680043.29396 50
1.2.826.0.1.3680043.29397 50
1.2.826.0.1.3680043.29409 50
1.2.826.0.1.3680043.29425 50
1.2.826.0.1.3680043.29457 50
1.2.826.0.1.3680043.29469 50
1.2.826.0.1.3680043.29539 50
1.2.826.0.1.3680043.29548 50
1.2.826.0.1.3680043.29571 50
1.2.826.0.1.3680043.29630 50
1.2.826.0.1.3680043.29631 50
1.2.826.0.1.3680043.29650 50
1.2.826.0.1.3680043.29668 50
1.2.826.0.1.3680043.29671 50
1.2.826.0.1.3680043.29747 50
1.2.826.0.1.3680043.29778 50
1.2.826.0.1.3680043.29791 50
1.2.826.0.1.3680043.29795 50
1.2.826.0.1.3680043.29804 50
1.2.826.0.1.3680043.29805 50
1.2.826.0.1.3680043.29807 50
1.2.826.0.1.3680043.29833 50
1.2.826.0.1.3680043.29837 50
1.2.826.0.1.3680043.29865 50
1.2.826.0.1.3680043.29872 50
1.2.826.0.1.3680043.29876 50
1.2.826.0.1.3680043.29878 50
1.2.826.0.1.3680043.29882 50
1.2.826.0.1.3680043.29889 50
1.2.826.0.1.3680043.29892 50
1.2.826.0.1.3680043.29924 50
1.2.826.0.1.3680043.29949 50
1.2.826.0.1.3680043.29952 50
1.2.826.0.1.3680043.29968 50
1.2.826.0.1.3680043.29986 50
1.2.826.0.1.3680043.29993 50
1.2.826.0.1.3680043.30020 50
1.2.826.0.1.3680043.30030 50
1.2.826.0.1.3680043.30037 50
1.2.826.0.1.3680043.30038 50
1.2.826.0.1.3680043.30039 50
1.2.826.0.1.3680043.30051 50
1.2.826.0.1.3680043.30055 50
1.2.826.0.1.3680043.30067 50
1.2.826.0.1.3680043.30089 50
1.2.826.0.1.3680043.30122 50
1.2.826.0.1.3680043.30134 50
1.2.826.0.1.3680043.30177 50
1.2.826.0.1.3680043.30185 50
1.2.826.0.1.3680043.30201 50
1.2.826.0.1.3680043.30213 50
1.2.826.0.1.3680043.30219 50
1.2.826.0.1.3680043.30221 50
1.2.826.0.1.3680043.30238 50
1.2.826.0.1.3680043.30240 50
1.2.826.0.1.3680043.30277 50
1.2.826.0.1.3680043.30282 50
1.2.826.0.1.3680043.30306 50
1.2.826.0.1.3680043.30307 50
1.2.826.0.1.3680043.30350 50
1.2.826.0.1.3680043.30366 50

【只切了一部分 — 全部列出】vol_id, 已有数, 应有数, 已有编号排序列表

【完全未切 — 全部列出】vol_id, 应有patch数
1.2.826.0.1.3680043.30382 50
1.2.826.0.1.3680043.30408 50
1.2.826.0.1.3680043.30437 50
1.2.826.0.1.3680043.30442 50
1.2.826.0.1.3680043.30451 50
1.2.826.0.1.3680043.30465 50
1.2.826.0.1.3680043.30475 50
1.2.826.0.1.3680043.30487 50
1.2.826.0.1.3680043.30498 50
1.2.826.0.1.3680043.30505 50
1.2.826.0.1.3680043.30509 50
1.2.826.0.1.3680043.30511 50
1.2.826.0.1.3680043.30515 50
1.2.826.0.1.3680043.30524 50
1.2.826.0.1.3680043.30539 50
1.2.826.0.1.3680043.30549 50
1.2.826.0.1.3680043.30565 50
1.2.826.0.1.3680043.30574 50
1.2.826.0.1.3680043.30604 50
1.2.826.0.1.3680043.30610 50
1.2.826.0.1.3680043.30640 50
1.2.826.0.1.3680043.30657 50
1.2.826.0.1.3680043.30682 50
1.2.826.0.1.3680043.30683 50
1.2.826.0.1.3680043.30708 50
1.2.826.0.1.3680043.30713 50
1.2.826.0.1.3680043.30731 50
1.2.826.0.1.3680043.30741 50
1.2.826.0.1.3680043.30765 50
1.2.826.0.1.3680043.30778 50
1.2.826.0.1.3680043.30787 50
1.2.826.0.1.3680043.30795 50
1.2.826.0.1.3680043.30797 50
1.2.826.0.1.3680043.30831 50
1.2.826.0.1.3680043.30864 50
1.2.826.0.1.3680043.30880 50
1.2.826.0.1.3680043.30899 50
1.2.826.0.1.3680043.30905 50
1.2.826.0.1.3680043.30911 50
1.2.826.0.1.3680043.30912 50
1.2.826.0.1.3680043.30914 50
1.2.826.0.1.3680043.30985 50
1.2.826.0.1.3680043.31002 50
1.2.826.0.1.3680043.31005 50
1.2.826.0.1.3680043.31017 50
1.2.826.0.1.3680043.31026 50
1.2.826.0.1.3680043.31040 50
1.2.826.0.1.3680043.31041 50
1.2.826.0.1.3680043.31067 50
1.2.826.0.1.3680043.31072 50
1.2.826.0.1.3680043.31077 50
1.2.826.0.1.3680043.31083 50
1.2.826.0.1.3680043.31088 50
1.2.826.0.1.3680043.31114 50
1.2.826.0.1.3680043.31151 50
1.2.826.0.1.3680043.31205 50
1.2.826.0.1.3680043.31245 50
1.2.826.0.1.3680043.31264 50
1.2.826.0.1.3680043.31328 50
1.2.826.0.1.3680043.31351 50
1.2.826.0.1.3680043.31352 50
1.2.826.0.1.3680043.31357 50
1.2.826.0.1.3680043.31368 50
1.2.826.0.1.3680043.31370 50
1.2.826.0.1.3680043.31379 50
1.2.826.0.1.3680043.31394 50
1.2.826.0.1.3680043.31409 50
1.2.826.0.1.3680043.31411 50
1.2.826.0.1.3680043.31419 50
1.2.826.0.1.3680043.31446 50
1.2.826.0.1.3680043.31450 50
1.2.826.0.1.3680043.31475 50
1.2.826.0.1.3680043.31532 50
1.2.826.0.1.3680043.31552 50
1.2.826.0.1.3680043.31569 50
1.2.826.0.1.3680043.31580 50
1.2.826.0.1.3680043.31581 50
1.2.826.0.1.3680043.31587 50
1.2.826.0.1.3680043.31595 50
1.2.826.0.1.3680043.31621 50
1.2.826.0.1.3680043.31629 50
1.2.826.0.1.3680043.31642 50
1.2.826.0.1.3680043.31643 50
1.2.826.0.1.3680043.31658 50
1.2.826.0.1.3680043.31681 50
1.2.826.0.1.3680043.31701 50
1.2.826.0.1.3680043.31718 50
1.2.826.0.1.3680043.31724 50
1.2.826.0.1.3680043.31738 50
1.2.826.0.1.3680043.31757 50
1.2.826.0.1.3680043.31841 50
1.2.826.0.1.3680043.31860 50
1.2.826.0.1.3680043.31900 50
1.2.826.0.1.3680043.31906 50
1.2.826.0.1.3680043.31907 50
1.2.826.0.1.3680043.31921 50
1.2.826.0.1.3680043.31977 50
1.2.826.0.1.3680043.31988 50
1.2.826.0.1.3680043.31995 50
1.2.826.0.1.3680043.32005 50
1.2.826.0.1.3680043.32017 50
1.2.826.0.1.3680043.32020 50
1.2.826.0.1.3680043.32023 50
1.2.826.0.1.3680043.32030 50
1.2.826.0.1.3680043.32046 50
1.2.826.0.1.3680043.32056 50
1.2.826.0.1.3680043.32063 50
1.2.826.0.1.3680043.32071 50
1.2.826.0.1.3680043.32076 50
1.2.826.0.1.3680043.32081 50
1.2.826.0.1.3680043.32151 50
1.2.826.0.1.3680043.32165 50
1.2.826.0.1.3680043.32190 50
1.2.826.0.1.3680043.32263 50
1.2.826.0.1.3680043.32280 50
1.2.826.0.1.3680043.32291 50
1.2.826.0.1.3680043.32303 50
1.2.826.0.1.3680043.32323 50
1.2.826.0.1.3680043.32357 50
1.2.826.0.1.3680043.32370 50
1.2.826.0.1.3680043.32387 50
1.2.826.0.1.3680043.32426 50
1.2.826.0.1.3680043.32428 50
1.2.826.0.1.3680043.32434 50
1.2.826.0.1.3680043.32436 50
1.2.826.0.1.3680043.32447 50
1.2.826.0.1.3680043.32458 50
1.2.826.0.1.3680043.32461 50
1.2.826.0.1.3680043.32476 50
1.2.826.0.1.3680043.32480 50
1.2.826.0.1.3680043.32534 50
1.2.826.0.1.3680043.32535 50
1.2.826.0.1.3680043.32577 50
1.2.826.0.1.3680043.32581 50
1.2.826.0.1.3680043.32585 50
1.2.826.0.1.3680043.32590 50
1.2.826.0.1.3680043.32597 50
1.2.826.0.1.3680043.32599 50
1.2.826.0.1.3680043.32627 50
1.2.826.0.1.3680043.32651 50
1.2.826.0.1.3680043.32658 50
1.2.826.0.1.3680043.32662 50
1.2.826.0.1.3680043.32692 50
1.2.826.0.1.3680043.32693 50
1.2.826.0.1.3680043.32721 50
1.2.826.0.1.3680043.32738 50
1.2.826.0.1.3680043.32754 50
1.2.826.0.1.3680043.32757 50
1.2.826.0.1.3680043.32766 50
1.2.826.0.1.3680043.32767 50
"""
# ================================================

UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip"
MISSING_ROOT = os.path.join(UNZIP_ROOT, "missing")


def parse_scan_root(text):
    m = re.search(r"SCAN_ROOT=(\S+)", text)
    if not m:
        raise ValueError("未在粘贴文本中找到 SCAN_ROOT=...")
    return m.group(1).rstrip("|")


def parse_vol_lines_in_section(text, section_title_keyword):
    """从某一节中解析每行: vol_id 与若干数字（只取行首的 vol_id）。"""
    # 找到以 section 标题开头的行，支持标题略有不同
    lines = text.splitlines()
    start = None
    for i, ln in enumerate(lines):
        if section_title_keyword in ln and "全部列出" in ln:
            start = i + 1
            break
    if start is None:
        return []
    out = []
    for ln in lines[start:]:
        ln = ln.strip()
        if not ln:
            continue
        if ln.startswith("【") and "】" in ln:
            break
        if ln.startswith("="):
            break
        # vol_id 与后面数字：取第一个连续 token 为 vol_id（Study UID 形如 1.2.826...）
        parts = ln.split()
        if len(parts) < 2:
            continue
        vol_id = parts[0]
        out.append(vol_id)
    return out


def find_volume_dir_or_file(scan_root, vol_id):
    """在 scan_root 下定位该体积：DICOM 叶子目录名 == vol_id，或 nii 路径对应的 vol_id 与收集逻辑一致。"""
    if not os.path.isdir(scan_root):
        return None, "scan_root 不存在"

    # 1) DICOM：叶子目录 basename == vol_id
    for dirpath, dirnames, filenames in os.walk(scan_root):
        if "__MACOSX" in dirpath.replace("\\", "/"):
            continue
        if os.path.basename(dirpath) != vol_id:
            continue
        dcms = [f for f in filenames if f.lower().endswith(".dcm")]
        if dcms and not dirnames:
            return ("dir", dirpath), None

    # 2) NIfTI：相对路径转下划线后等于 vol_id
    for dirpath, _, files in os.walk(scan_root):
        if "segmentations" in dirpath.replace("\\", "/").split("/"):
            continue
        for f in files:
            if not (f.endswith(".nii") or f.endswith(".nii.gz")):
                continue
            full = os.path.join(dirpath, f)
            rel = os.path.relpath(full, scan_root).replace("\\", "/")
            vid = rel.replace("/", "_").replace(".nii.gz", "").replace(".nii", "")
            if vid == vol_id:
                return ("file", full), None

    return None, "未找到"


def copy_volume(scan_root, vol_id, missing_root):
    kind_path, err = find_volume_dir_or_file(scan_root, vol_id)
    if kind_path is None:
        return False, err
    kind, path = kind_path
    rel = os.path.relpath(path, scan_root)
    dest = os.path.join(missing_root, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if kind == "dir":
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(path, dest, dirs_exist_ok=False)
    else:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy2(path, dest)
    return True, dest


# ---- 主流程 ----
scan_root = parse_scan_root(PASTE)
os.makedirs(MISSING_ROOT, exist_ok=True)

missing_none = parse_vol_lines_in_section(PASTE, "完全未切")
# 可选：只切了一部分
missing_partial = parse_vol_lines_in_section(PASTE, "只切了一部分")

# 去重，先未切再部分（部分优先保留一次）
seen = set()
vol_ids = []
for v in missing_none + missing_partial:
    if v in seen:
        continue
    seen.add(v)
    vol_ids.append(v)

print(f"SCAN_ROOT = {scan_root}")
print(f"MISSING_ROOT = {MISSING_ROOT}")
print(f"待复制 vol_id 数（完全未切 + 只切一部分，去重）: {len(vol_ids)}")

ok, bad = 0, []
for vid in vol_ids:
    success, info = copy_volume(scan_root, vid, MISSING_ROOT)
    if success:
        ok += 1
        print(f"OK  {vid} -> {info}")
    else:
        bad.append((vid, info))
        print(f"FAIL {vid}: {info}")

print("=" * 60)
print(f"成功复制: {ok} / {len(vol_ids)}")
if bad:
    print(f"失败 {len(bad)} 个:")
    for vid, msg in bad:
        print(f"  {vid}: {msg}")

SCAN_ROOT = /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/train_part_3
MISSING_ROOT = /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/missing
待复制 vol_id 数（完全未切 + 只切一部分，去重）: 150
OK  1.2.826.0.1.3680043.30382 -> /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/missing/train_part_3/1.2.826.0.1.3680043.30382
OK  1.2.826.0.1.3680043.30408 -> /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/missing/train_part_3/1.2.826.0.1.3680043.30408
OK  1.2.826.0.1.3680043.30437 -> /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/missing/train_part_3/1.2.826.0.1.3680043.30437
OK  1.2.826.0.1.3680043.30442 -> /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/missing/train_part_3/1.2.826.0.1.3680043.30442
OK  1.2.826.0.1.3680043.30451 -> /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/unzip/missing/train_part_3/1.2.826.0.1.3680043.30451
OK  1.2.826.0.1.3680043.30465 -> /content/drive/MyDrive/dataset/pretrain/RSNA-2022-CSFD/un